In [2]:
%pip install nrlmsise00

Note: you may need to restart the kernel to use updated packages.


In [61]:

import pandas as pd

url = "https://celestrak.org/SpaceData/SW-All.csv"

data = pd.read_csv(url)

aug15 = data[data["DATE"] == "2026-08-15"].iloc[0]
aug16 = data[data["DATE"] == "2026-08-16"].iloc[0]

f107 = float(aug15["F10.7_OBS"])
f107a = float(aug16["F10.7_OBS_CENTER81"])
ap = float(aug16["AP_AVG"])

print("F107:", f107)
print("F107A:", f107a)
print("Ap:", ap)


F107: 117.1
F107A: 117.1
Ap: 3.0


In [68]:

import numpy as np
from datetime import datetime
from nrlmsise00 import msise_model

# Constants
D = 0.5
m = 400
g = 9.81

A = np.pi * (D / 2)**2

# I picked Russellville on august 16th 2026 
date = datetime(2026, 8, 16, 17, 10)
lat = 35.2784
lon = -93.1338


# Atmospheric density and temp
def rho(y):
    densities, temperatures = msise_model(
        date, y / 1000, lat, lon,  f107a, f107, ap
    )
    return densities[5] * 1000

def T(y):
    densities, temperatures = msise_model(
        date, y / 1000, lat, lon, f107a, f107, ap
    )
    return temperatures[1]

# Peice wise function of drage coeffiecnt 
def Cd(M):
    if M <= 0.5:
        return 0.26
    elif M <= 1:
        return 0.4
    elif M <= 1.5:
        return 0.55
    elif M <= 2:
        return 0.47
    elif M <= 2.5:
        return 0.36
    elif M <= 3:
        return 0.28
    elif M <= 3.5:
        return 0.24
    elif M <= 4:
        return 0.21
    elif M <= 4.5:
        return 0.2
    elif M <= 5:
        return 0.2
    elif M <= 5.5:
        return 0.2
    elif M <= 6:
        return 0.21
    elif M <= 6.5:
        return 0.23
    elif M <= 7:
        return 0.24
    elif M <= 7.5:
        return 0.25
    else:
        return 0.26

# Differential equations
def mach(v, y):
    gamma = 1.4
    R = 287

    c = np.sqrt(gamma * R * T(y))

    return abs(v) / c


def equations(t, state):
    y, v = state

    dydt = v
    M = mach(v, y)

    dvdt = -g - (Cd(M) * rho(y) * A * v * abs(v)) / (2 * m)

    return [dydt, dvdt]


# Initial conditions
y = 0
v = 7599.68
t = 0

# time intervals of 0.5 seconds 
dt = 0.01

# set to end loop when rocket stops moving up
while v > 0:

    c = np.sqrt(1.4 * 287 * T(y))

    M = v / c

    F_drag = 0.5 * Cd(M) * rho(y) * A * v**2

    a = -g - F_drag / m

    y = y + v * dt

    v = v + a * dt

    t = t + dt

    print("Time:", t, "s")
    print("Altitude:", y / 1000, "km")
    print("Velocity:", v, "m/s")
    print("--------------------")

Time: 0.01 s
Altitude: 0.0759968 km
Velocity: 7555.415912849911 m/s
--------------------
Time: 0.02 s
Altitude: 0.15155095912849914 km
Velocity: 7512.009914349103 m/s
--------------------
Time: 0.03 s
Altitude: 0.22667105827199016 km
Velocity: 7469.4358531683765 m/s
--------------------
Time: 0.04 s
Altitude: 0.3013654168036739 km
Velocity: 7427.668674806915 m/s
--------------------
Time: 0.05 s
Altitude: 0.37564210355174305 km
Velocity: 7386.684363513216 m/s
--------------------
Time: 0.060000000000000005 s
Altitude: 0.44950894718687523 km
Velocity: 7346.459887902607 m/s
--------------------
Time: 0.07 s
Altitude: 0.5229735460659013 km
Velocity: 7306.973149997452 m/s
--------------------
Time: 0.08 s
Altitude: 0.5960432775658757 km
Velocity: 7268.202937439248 m/s
--------------------
Time: 0.09 s
Altitude: 0.6687253069402683 km
Velocity: 7230.1288786426985 m/s
--------------------
Time: 0.09999999999999999 s
Altitude: 0.7410265957266953 km
Velocity: 7192.73140068087 m/s
--------------

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Time: 373.3899999998054 s
Altitude: 776.3055452932787 km
Velocity: 236.00469731675958 m/s
--------------------
Time: 373.3999999998054 s
Altitude: 776.3079053402519 km
Velocity: 235.9065973167596 m/s
--------------------
Time: 373.4099999998054 s
Altitude: 776.3102644062251 km
Velocity: 235.8084973167596 m/s
--------------------
Time: 373.4199999998054 s
Altitude: 776.3126224911982 km
Velocity: 235.7103973167596 m/s
--------------------
Time: 373.4299999998054 s
Altitude: 776.3149795951714 km
Velocity: 235.61229731675962 m/s
--------------------
Time: 373.43999999980537 s
Altitude: 776.3173357181446 km
Velocity: 235.51419731675963 m/s
--------------------
Time: 373.44999999980536 s
Altitude: 776.3196908601177 km
Velocity: 235.41609731675965 m/s
--------------------
Time: 373.45999999980535 s
Altitude: 776.3220450210908 km
Velocity: 235.31799731675966 m/s
--------------------
Time: 373.46999999980534 s
Altitude: 776.3243982010641 km
Velocity: 235.21989731675967 m/s
--------------------


In [69]:
# Checking V0 of 2000 m/s

import numpy as np
from datetime import datetime
from nrlmsise00 import msise_model

D = 0.5
m = 400
g = 9.81

A = np.pi * (D / 2)**2

date = datetime(2026, 8, 16, 17, 10)
lat = 35.2784
lon = -93.1338

def rho(y):
    densities, temperatures = msise_model(
        date, y / 1000, lat, lon,  f107a, f107, ap
    )
    return densities[5] * 1000

def T(y):
    densities, temperatures = msise_model(
        date, y / 1000, lat, lon, f107a, f107, ap
    )
    return temperatures[1]

def Cd(M):
    if M <= 0.5:
        return 0.26
    elif M <= 1:
        return 0.4
    elif M <= 1.5:
        return 0.55
    elif M <= 2:
        return 0.47
    elif M <= 2.5:
        return 0.36
    elif M <= 3:
        return 0.28
    elif M <= 3.5:
        return 0.24
    elif M <= 4:
        return 0.21
    elif M <= 4.5:
        return 0.2
    elif M <= 5:
        return 0.2
    elif M <= 5.5:
        return 0.2
    elif M <= 6:
        return 0.21
    elif M <= 6.5:
        return 0.23
    elif M <= 7:
        return 0.24
    elif M <= 7.5:
        return 0.25
    else:
        return 0.26

def mach(v, y):
    gamma = 1.4
    R = 287

    c = np.sqrt(gamma * R * T(y))

    return abs(v) / c


def equations(t, state):
    y, v = state

    dydt = v
    M = mach(v, y)

    dvdt = -g - (Cd(M) * rho(y) * A * v * abs(v)) / (2 * m)

    return [dydt, dvdt]


y = 0
v = 2000
t = 0

dt = 0.01

while v > 0:

    c = np.sqrt(1.4 * 287 * T(y))

    M = v / c

    F_drag = 0.5 * Cd(M) * rho(y) * A * v**2

    a = -g - F_drag / m

    y = y + v * dt

    v = v + a * dt

    t = t + dt

    print("Time:", t, "s")
    print("Altitude:", y / 1000, "km")
    print("Velocity:", v, "m/s")
    print("--------------------")

Time: 0.01 s
Altitude: 0.02 km
Velocity: 1997.4312952718237 m/s
--------------------
Time: 0.02 s
Altitude: 0.039974312952718236 km
Velocity: 1994.874078784221 m/s
--------------------
Time: 0.03 s
Altitude: 0.05992305374056045 km
Velocity: 1992.3282674685815 m/s
--------------------
Time: 0.04 s
Altitude: 0.07984633641524627 km
Velocity: 1989.7937790926285 m/s
--------------------
Time: 0.05 s
Altitude: 0.09974427420617255 km
Velocity: 1987.2705322496975 m/s
--------------------
Time: 0.060000000000000005 s
Altitude: 0.11961697952866954 km
Velocity: 1984.7584463481826 m/s
--------------------
Time: 0.07 s
Altitude: 0.13946456399215137 km
Velocity: 1982.2574416011457 m/s
--------------------
Time: 0.08 s
Altitude: 0.15928713840816283 km
Velocity: 1979.7674390160855 m/s
--------------------
Time: 0.09 s
Altitude: 0.1790848127983237 km
Velocity: 1977.288360384865 m/s
--------------------
Time: 0.09999999999999999 s
Altitude: 0.19885769640217232 km
Velocity: 1974.8201282737932 m/s
-------

In [80]:
# Checking V0 of 2025.5 m/s

import numpy as np
from datetime import datetime
from nrlmsise00 import msise_model

D = 0.5
m = 400
g = 9.81

A = np.pi * (D / 2)**2

date = datetime(2026, 8, 16, 17, 10)
lat = 35.2784
lon = -93.1338

def rho(y):
    densities, temperatures = msise_model(
        date, y / 1000, lat, lon,  f107a, f107, ap
    )
    return densities[5] * 1000

def T(y):
    densities, temperatures = msise_model(
        date, y / 1000, lat, lon, f107a, f107, ap
    )
    return temperatures[1]

def Cd(M):
    if M <= 0.5:
        return 0.26
    elif M <= 1:
        return 0.4
    elif M <= 1.5:
        return 0.55
    elif M <= 2:
        return 0.47
    elif M <= 2.5:
        return 0.36
    elif M <= 3:
        return 0.28
    elif M <= 3.5:
        return 0.24
    elif M <= 4:
        return 0.21
    elif M <= 4.5:
        return 0.2
    elif M <= 5:
        return 0.2
    elif M <= 5.5:
        return 0.2
    elif M <= 6:
        return 0.21
    elif M <= 6.5:
        return 0.23
    elif M <= 7:
        return 0.24
    elif M <= 7.5:
        return 0.25
    else:
        return 0.26

def mach(v, y):
    gamma = 1.4
    R = 287

    c = np.sqrt(gamma * R * T(y))

    return abs(v) / c


def equations(t, state):
    y, v = state

    dydt = v
    M = mach(v, y)

    dvdt = -g - (Cd(M) * rho(y) * A * v * abs(v)) / (2 * m)

    return [dydt, dvdt]



y = 0
v = 2025.5
t = 0

dt = 0.01

while v > 0:

    c = np.sqrt(1.4 * 287 * T(y))

    M = v / c

    F_drag = 0.5 * Cd(M) * rho(y) * A * v**2

    a = -g - F_drag / m

    y = y + v * dt

    v = v + a * dt

    t = t + dt

    print("Time:", t, "s")
    print("Altitude:", y / 1000, "km")
    print("Velocity:", v, "m/s")
    print("--------------------")

Time: 0.01 s
Altitude: 0.020255 km
Velocity: 2022.8678932235741 m/s
--------------------
Time: 0.02 s
Altitude: 0.04048367893223574 km
Velocity: 2020.2477131414728 m/s
--------------------
Time: 0.03 s
Altitude: 0.06068615606365047 km
Velocity: 2017.639372438993 m/s
--------------------
Time: 0.04 s
Altitude: 0.0808625497880404 km
Velocity: 2015.0427846914977 m/s
--------------------
Time: 0.05 s
Altitude: 0.10101297763495537 km
Velocity: 2012.4578643528646 m/s
--------------------
Time: 0.060000000000000005 s
Altitude: 0.12113755627848402 km
Velocity: 2009.884526744116 m/s
--------------------
Time: 0.07 s
Altitude: 0.14123640154592518 km
Velocity: 2007.3226880422278 m/s
--------------------
Time: 0.08 s
Altitude: 0.16130962842634747 km
Velocity: 2004.7722652691111 m/s
--------------------
Time: 0.09 s
Altitude: 0.18135735107903858 km
Velocity: 2002.2331762807687 m/s
--------------------
Time: 0.09999999999999999 s
Altitude: 0.20137968284184626 km
Velocity: 1999.7053397566171 m/s
----